# Chapter 3
So far we have only worked for continuous data, but when dealing with **evoked design\*** we have to create epochs meaning we have to slice the continuous data into small time segments (1-2 sec long) to analyse them.

Therefore, in this chapter we will focus on creating epochs to avarage these epochs to generate evoked responses (ERP/ERF)*.

> \* In the context of electroencephalography (EEG), "evoked design" refers to the experimental methodology and analysis techniques used to measure evoked potentials (EPs) or event-related potentials (ERPs). These are the specific, time-locked electrical signals generated by the brain in response to a controlled external stimulus (sensory, motor, or cognitive event), as distinct from the brain's ongoing, spontaneous electrical activity (the background EEG). 
>
> \* In Electroencephalogram (EEG), ERP (Event-Related Potential) refers to tiny, time-locked voltage changes in the brain's electrical activity (measured by EEG) that occur in response to specific sensory, cognitive, or motor events, revealing the timing of mental processes, while ERF (Event-Related Field) is the equivalent magnetic signal measured with Magnetoencephalography (MEG), reflecting the same underlying neural activity. Both techniques average many trials to filter out background noise, isolating these subtle, event-specific brain responses (like the P300 or N400) to understand perception, attention, and disorders. 

## Libraries & Config

In [ ]:
import pathlib
import matplotlib

import mne

matplotlib.use('QtAgg')

In [ ]:
import mne_bids

## Loading, filtering and extracting the events out of the BIDS data.

Loading the BIDS data:

In [ ]:
bids_root = pathlib.Path('./out_data/sample_BIDS')

bids_path = mne_bids.BIDSPath(
    subject='01',
    session='01',
    task='audiovisual',
    run='01',
    datatype='meg',
    root=bids_root,
)

raw_bids_data = mne_bids.read_raw_bids(bids_path)

Filtering the data (make sure to load the whole data first):

In [ ]:
raw_bids_data.load_data()  # load the data into memory

raw_bids_data.filter(l_freq=1, h_freq=40) # band-pass filter between 1 and 40 Hz

Extracting the events:

In [ ]:
events, event_id = mne.events_from_annotations(
    raw_bids_data
)

event_id

## Creating and visualising Epochs

Now that we have the filtered raw data and the accompanying events we can focos on creating epochs. However, for that we need to specify few specifications about the epoch such as:
+ Where does the epoch starts **relative to an event onset**, i.e., `tmin` (we specify time in sec).
+ Where does the epoch ends **relative to an event onset**, i.e., `tmax` we specify time in sec).
+ Should we apply some sort of **baseline correction\***.

> Baseline correction in EEG epochs is a preprocessing step that removes slow, unwanted voltage shifts (drifts/artifacts) from EEG data by subtracting the average EEG activity from a predefined "baseline" time window (before the event) from the entire epoch, ensuring that event-related responses (ERPs) are measured relative to zero or a consistent level, making them easier to compare and analyze. It effectively centers each trial's signal, isolating the brain's response to a stimulus from general background noise or slow electrode changes. 

In [ ]:
tmin = -0.3  # start of each epoch (300ms before the trigger)
tmax = 0.5   # end of each epoch (500ms after the trigger)

# baseline correction period (from the first instant to t=0), i.e., pre-stimulus period.
# It is basically a tuple with two values: (start, end) in seconds
# Here None means from the beginning of the epoch, and
# 0 means the time of the trigger/event starts
# i.e., all together it means from the start of the epoch to the time of the trigger/event starts
basline = (None, 0)  

Now to create the epochs we can initialise the `mne.Epochs` class and pass the *bids* data, *events* info and all the specifications to it:

In [ ]:
epochs = mne.Epochs(
    raw=raw_bids_data,
    events=events, # events to use to cut the epochs
    event_id=event_id,
    tmin=tmin,
    tmax=tmax,
    baseline=basline, # optional, with None meaning no baseline correction
    preload=True # load the epochs into memory
)

epochs

To visualise the epochs we call simply call its `.plot()` function:

In [ ]:
epochs.plot()

Let's now create epochs starting 250 ms before the stimulus onset and ending 800 ms after stimulus onset, and apply baseline correctin with a baseline period ranging from -200 to 0 ms.

In [ ]:
tmin = -0.25  # start of each epoch (250ms before the trigger)
tmax = 0.8   # end of each epoch (800ms after the trigger)

basline = (-0.2, 0)  # baseline correction period (from -200ms to the time of the trigger/event starts)

epochs_new = mne.Epochs(
    raw=raw_bids_data,
    events=events, # events to use to cut the epochs
    event_id=event_id,
    tmin=tmin,
    tmax=tmax,
    baseline=basline, # optional, with None meaning no baseline correction
    preload=True # load the epochs into memory
)

epochs_new

In [ ]:
epochs_new.plot()

# Selecting epochs based on events

If we only wants the epochs of a specific event, we can simpy call for it by:

In [ ]:
event_id

In [ ]:
epochs['Auditory/Left']

we can also combine subsets, like, if we want to combine all the Auditory events in one go we can simply call:

In [ ]:
epochs['Auditory'].plot()

This happens because *mne* follows a pattern of putting `/` when specifying a subugroup and therefore it can use that info to find the group or subgroup.

Furthermore, we can also visualise these epoch in a different way other than the `.plot()` method by calling the `.plot_image()` func, which generate a ERP image*.

> An ERP (Event-Related Potential) image in EEG is a powerful 2D visualization that displays single-trial brain responses as colored horizontal lines, stacked vertically, to reveal hidden patterns beyond the traditional averaged ERP; these trials are often sorted by reaction time or other factors, allowing researchers to see trial-to-trial variability, detect noise, and understand underlying neural dynamics that get lost in simple averages, using color to map voltage/amplitude over time. 

In [ ]:
epochs['Visual'].plot_image()

Let's do the same for all the epochs with "Right" conditions and then plot the ERP image for the EEG channels.

In [ ]:
# (epochs["Right"]
# .copy()
# .pick(exclude=[], picks=["eeg"])
# .plot_image())

epochs['Right'].plot_image(picks=['eeg'])

## Saving epochs

To save the epochs we call its `.save` func:

In [ ]:
epochs.save(
    fname=pathlib.Path('out_data') / 'epochs-epo.fif',
    overwrite=True
)

## Creating and plotting evoked data

Evoked data is really just an average of epochs of a particular event and we can get it by simply calling the `.average()` function of a particular event epochs data.

> Evoked data, or evoked potentials (EPs), in an EEG setting are specific electrical signals from the nervous system that occur in response to a controlled external sensory stimulus (visual, auditory, or somatosensory). They are highly useful for diagnosing neurological conditions, localizing damage in nerve pathways, and monitoring nerve function during surgery. 

In [ ]:
evoked_auditory = epochs['Auditory'].average()
evoked_visual = epochs['Visual'].average()

evoked_auditory

To plot the evoked data we call:
+ `.plot()` - readings traces map
+ `.plot_topomap()` -  a topographic maps of a particular channel type
+ `.plot_joint()` -  a joint map of traces and topo maps

All these plots help us in getting a better understanding of **Global Field Power (GFP)\***

> * In EEG, GFP stands for Global Field Power, a crucial reference-independent measure that quantifies the overall strength or amount of electrical activity across all scalp electrodes at any given moment, essentially showing the spatial extent/variability of the brain's electrical field, with peaks indicating moments of significant, stable cortical activation used to time event-related potentials (ERPs) and analyze EEG states like microstates. 

In [ ]:
# to plot each channel in a different color we use spatial_colors=True, because
# this way it tell us MNE to use a colormap to color the channels based on their spatial location
evoked_auditory.plot(spatial_colors=True)

In [ ]:
# by default the topomaps picks a 4 time points to plot the topographic maps, otherwise
# we can specify the time points using the 'times' parameter as a list of time points in seconds
evoked_auditory.plot_topomap(ch_type="mag")

In [ ]:
evoked_visual.plot_topomap(ch_type="mag", times=[0, 0.05, 0.075, 0.0, 0.1, 0.125, 0.15, 0.2])

In [ ]:
evoked_auditory.plot_topomap(ch_type="eeg", times=[0, 0.05, 0.075, 0.0, 0.1, 0.125, 0.15, 0.2])

In [ ]:
evoked_auditory.plot_joint(picks='mag')

To plot *GFPs* for different conditions/subgroup of an event to help us contrast them and it can be done by using `mne.viz.plot_compare_evokeds()`

In [ ]:
mne.viz.plot_compare_evokeds(
    evokeds=[evoked_auditory, evoked_visual],
    picks='mag'
)

Let's us now plot a GFP comparison for the "Visual/Left" and "Visual/Right" conditions of the EEG data.

In [ ]:
evoked_visual_left = epochs["Visual/Left"].average()
evoked_visual_right = epochs["Visual/Right"].average()

mne.viz.plot_compare_evokeds(
    evokeds=[evoked_visual_left, evoked_visual_right],
    picks="eeg"
)

## Saving and Reading evoked data

To save the evoked data we can call the `mne.wrote_evokeds()` func:

In [ ]:
mne.write_evokeds(
    fname=pathlib.Path("out_data") / "evokeds-ave.fif",
    evoked=[evoked_auditory, evoked_visual]
)

And to read the evokeds data we can call the `mne.read_evokeds()`:

> Note: One can also use the `mne.read_epochs_eeglab()` to read epochs data that has already preproceessed with another tool.

In [ ]:
evokeds = mne.read_evokeds(
    fname=pathlib.Path("out_data") / "evokeds-ave.fif",
)

evokeds

As we can see at the end of the output we get a list of two evoked and we can access them by:

In [ ]:
evokeds[0]

We can also have a condition when reading a evokeds file which contains multiple evokeds and we  only want a very specific evoked and we know what is called, in that case we can pass the `condition` param with the condition name:

In [ ]:
evoked_conditional = mne.read_evokeds(
    fname=pathlib.Path("out_data") / "evokeds-ave.fif",
    condition="0.50 × Auditory/Left + 0.50 × Auditory/Right"
)

evoked_conditional

# The End & Have a great day!